# ImageEval 2026 — Task 1b · Run HP2: Beam Search (num_beams=4)

**Hypothesis:** Greedy decoding commits to the single highest-probability token at each step.
Beam search (num_beams=4) keeps 4 candidates alive and picks the globally best sequence.
For the answer-first format, the first token is the prediction digit — beam search prevents
committing to a suboptimal answer due to a local probability peak.

Everything else is identical to Run 4 (answer-first, Qwen2.5-VL-7B-Instruct, NF4 4-bit).


## 1. Install & Kaggle compatibility

In [ ]:
import os
for _major in ('12', '13'):
    _src = f'/usr/local/cuda/lib64/libnvJitLink.so.{_major}'
    _dst = '/usr/local/cuda/lib64/libnvJitLink.so.13'
    if os.path.exists(_src) and not os.path.exists(_dst):
        os.symlink(_src, _dst)
        print(f'Symlinked libnvJitLink .{_major} -> .13')
        break
os.environ['BITSANDBYTES_NOWELCOME'] = '1'
os.environ['BNB_CUDA_VERSION'] = '128'
!pip install -q -U 'transformers>=4.49.0' accelerate bitsandbytes qwen-vl-utils 2>&1 | tail -5
print('Dependencies ready.')


## 2. Configuration

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

REPO_ID    = 'QCRI/AynVQA-ArabicNLP26'
TASK       = 'task1b'
LANG       = 'en'
SPLIT      = 'devtest'   # change to 'dev' to get a local score
MAX_ITEMS  = None

RUN_ID     = 'hp2_beam_search_4'  # used in output filenames

VLM_MODEL  = 'Qwen/Qwen2.5-VL-7B-Instruct'
QUANTIZE   = True          # NF4 4-bit -> fits T4 16GB

MAX_PIXELS     = 1024 * 28 * 28
MAX_NEW_TOKENS = 256       # answer on first line + justification
NUM_BEAMS      = 4               # beam search width

print(f'Run: {RUN_ID}')
print(f'config: {TASK}_{LANG}/{SPLIT} | {VLM_MODEL} | quantized={QUANTIZE}')
print(f'MAX_PIXELS={MAX_PIXELS} | MAX_NEW_TOKENS={MAX_NEW_TOKENS}')
print('Estimated inference time: ~45 min on T4 (~10% overhead from beam tracking)')


In [ ]:
from huggingface_hub import login
login(token=os.environ.get('HF_TOKEN', 'hf_YOUR_TOKEN_HERE'))

## 3. Download split + images

In [ ]:
import json
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

jsonl = hf_hub_download(REPO_ID, filename=f'{TASK}/{SPLIT}_{LANG}.jsonl', repo_type='dataset')
records = [json.loads(l) for l in open(jsonl, encoding='utf-8') if l.strip()]
if MAX_ITEMS:
    records = records[:MAX_ITEMS]
print(len(records), 'items; labelled:', 'labels' in records[0])

needed = sorted({r['image'] for r in records})
paths = {}
for rel in tqdm(needed, desc='images'):
    try:
        paths[rel] = hf_hub_download(REPO_ID, filename=rel, repo_type='dataset')
    except Exception as e:
        print(f'Failed: {rel}: {e}')
print(f'Downloaded {len(paths)}/{len(needed)} images.')


## 4. Load model

In [ ]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info

assert torch.cuda.is_available(), 'No GPU — change runtime to T4.'
print('GPU:', torch.cuda.get_device_name(0))
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

quant = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=dtype, bnb_4bit_use_double_quant=True,
) if QUANTIZE else None

processor = AutoProcessor.from_pretrained(VLM_MODEL, max_pixels=MAX_PIXELS)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    VLM_MODEL, torch_dtype=dtype, device_map='auto',
    quantization_config=quant).eval()
print('Model loaded.')


## 5. True/False parser (verbatim from scorer)

In [ ]:
# --- True/False parser: verbatim copy of the official backbone.evaluate_tf ---
# This is what the Codabench 1b scorer uses, so local scores match exactly.
import re
from dataclasses import dataclass
from typing import Optional

TRUE_TOKENS = [r"\btrue\b", r"\byes\b", r"\bصح\b", r"\bصحيح\b", r"\bصحيحة\b",
               r"\bالصحيح\b", r"\bالخطأ\b"]
FALSE_TOKENS = [r"\bfalse\b", r"\bfalsy\b", r"\bno\b", r"\bخطأ\b", r"\bالخطأ\b",
                r"\bغلط\b", r"\bغير\s+صحيح(?:ة)?\b", r"\bغير\s+صحيحة\b",
                r"\bخاطئ\b", r"\bخاطئة\b"]
ABSTAIN_PATTERNS = [
    r"\b(can(?:not|'t)\s+determine|can(?:not|'t)\s+tell|not\s+enough\s+information|cannot\s+be\s+sure|unclear)\b",
    r"لا\s+يمكن(?:نا)?\s+الجزم", r"لا\s+يمكن\s+الجزم",
    r"لا\s+يمكن\s+تحديد.*(?:صحة|خطأ|صحيح|خاطئ|العبارة)",
    r"لا\s+نستطيع\s+التأكد", r"لا\s+يمكن\s+الحكم"]
STRONG_CUES = [
    r"therefore[,:\s]*", r"final\s+answer[,:\s]*", r"the\s+answer\s+is[,:\s]*",
    r"correct\s+answer\s+is[,:\s]*", r"so\s+the\s+answer\s+is[,:\s]*",
    r"conclusion[,:\s]*", r"verdict[,:\s]*", r"determination[,:\s]*",
    r"final[,:\s]*(?:answer)?[,:\s]*", r"the\s+statement\s+is\s*[:\-–—,]?\s*",
    r"الإجابة\s+الصحيحة\s*(?:هي)?\s*[:：]?\s*",
    r"الجواب\s+الصحيح\s*(?:هو|هي)?\s*[:：]?\s*",
    r"الإجابة\s*(?:هي)?\s*[:：]?\s*", r"الجواب\s*(?:هو|هي)?\s*[:：]?\s*",
    r"إذًا\s*(?:الجواب|الجواب\s+هو|الإجابة|الإجابة\s+هي)?\s*[:：]?\s*",
    r"الإجابة\s+النهائية\s*(?:هي)?\s*[:：]?\s*"]


@dataclass
class EvalResult:
    pred: Optional[str]
    confidence: float
    needs_review: bool
    reason: str
    conflict: bool


def _normalize(text):
    t = (text or "").strip().replace("‏", "").replace("‎", "").lower()
    return re.sub(r"[ \t]+", " ", t)


def _strip_code_and_quotes(text):
    t = text or ""
    t = re.sub(r"```.*?```", " ", t, flags=re.DOTALL)
    t = re.sub(r"\".*?\"", " ", t, flags=re.DOTALL)
    t = re.sub(r"“.*?”", " ", t, flags=re.DOTALL)
    return t


def _match_label(fragment):
    for pat in TRUE_TOKENS:
        if re.search(pat, fragment, flags=re.IGNORECASE):
            return "true"
    for pat in FALSE_TOKENS:
        if re.search(pat, fragment, flags=re.IGNORECASE):
            return "false"
    return None


def _has_any(patterns, text):
    return any(re.search(p, text, flags=re.IGNORECASE) for p in patterns)


def evaluate_tf(response):
    raw = response or ""
    if not raw.strip():
        return EvalResult(None, 0.0, True, "empty_response", conflict=False)
    text_noquotes = _normalize(_strip_code_and_quotes(raw))

    first_label_pos = first_label_value = None
    for pat, lab in [(p, "true") for p in TRUE_TOKENS] + [(p, "false") for p in FALSE_TOKENS]:
        m = re.search(pat, text_noquotes, flags=re.IGNORECASE)
        if m and (first_label_pos is None or m.start() < first_label_pos):
            first_label_pos, first_label_value = m.start(), lab
    first_abstain_pos = None
    for pat in ABSTAIN_PATTERNS:
        m = re.search(pat, text_noquotes, flags=re.IGNORECASE)
        if m and (first_abstain_pos is None or m.start() < first_abstain_pos):
            first_abstain_pos = m.start()
    if first_label_pos is not None or first_abstain_pos is not None:
        if first_abstain_pos is not None and (first_label_pos is None or first_abstain_pos < first_label_pos):
            return EvalResult(None, 0.0, True, "abstain_before_label", conflict=False)
        if first_label_pos is not None and (first_abstain_pos is None or first_label_pos < first_abstain_pos):
            return EvalResult(first_label_value, 0.95, False, "first_explicit_label", conflict=False)

    best = None
    m0 = re.match(r"^\s*[\*\s_`]*((?:true|false)|(?:صح|صحيح|صحيحة)|(?:خطأ|غلط|خاطئ|خاطئة))\b",
                  text_noquotes, flags=re.IGNORECASE)
    if m0:
        label = _match_label(m0.group(1))
        if label:
            best = (3.0, label, "leading_label")
    for cue in STRONG_CUES:
        for m in re.finditer(cue, text_noquotes, flags=re.IGNORECASE):
            label = _match_label(text_noquotes[m.end():m.end() + 140])
            if label:
                cand = (3.0, label, "strong_cue")
                best = max(best, cand, key=lambda x: x[0]) if best else cand
    lines = [ln.strip() for ln in raw.splitlines() if ln.strip()]
    standalone = [
        (re.compile(r"^[\*\s_`]*true\s*[\.!\?]*[\*\s_`]*$", re.IGNORECASE), "true"),
        (re.compile(r"^[\*\s_`]*false\s*[\.!\?]*[\*\s_`]*$", re.IGNORECASE), "false"),
        (re.compile(r"^[\*\s_`]*صح\s*[\.!\?]*[\*\s_`]*$"), "true"),
        (re.compile(r"^[\*\s_`]*(خطأ|غلط|خاطئ|خاطئة)\s*[\.!\?]*[\*\s_`]*$"), "false")]
    if not best or best[0] < 3.0:
        for ln in reversed(lines[-25:]):
            ln_norm = _normalize(ln)
            for rgx, lab in standalone:
                if rgx.match(ln_norm):
                    best = (2.0, lab, "standalone_label_line")
                    break
            if best:
                break
    if not best:
        tail = text_noquotes[-450:]
        matches = []
        for pat in TRUE_TOKENS:
            matches += [(mm.start(), "true") for mm in re.finditer(pat, tail, flags=re.IGNORECASE)]
        for pat in FALSE_TOKENS:
            matches += [(mm.start(), "false") for mm in re.finditer(pat, tail, flags=re.IGNORECASE)]
        if matches:
            matches.sort(key=lambda x: x[0])
            best = (1.0, matches[-1][1], "last_occurrence_in_tail")
    if not best:
        return EvalResult(None, 0.0, True, "no_label_found", conflict=False)
    score, label, reason = best
    conflict = _has_any(TRUE_TOKENS, text_noquotes) and _has_any(FALSE_TOKENS, text_noquotes)
    confidence = {3.0: 0.95, 2.0: 0.80, 1.0: 0.60}.get(score, 0.50)
    return EvalResult(label, confidence, conflict and score <= 1.0, reason, conflict)

## 6. Answer-first joint inference

**Run 2 prompt order:** instructions → statements → `Answer: X` on last line.

**Run 4 prompt order:** instructions → statements → `Answer: X` on **first line** → justification.

The model must commit to a choice before rationalising it. This prevents the reasoning chain from drifting toward a culturally plausible but visually ungrounded statement before settling on a final answer (the mechanism M²CQA identify as driving elevated CFHR).

Everything else — fallback logic, parser, scorer — is identical to Run 2.


In [ ]:
import re

# ── Run 4: Answer-FIRST joint prompt ─────────────────────────────────────────
# Key difference from Run 2: the model writes 'Answer: X' on the FIRST line,
# then justifies. In Run 2 the answer came last. This is the only change.
JOINT_PROMPT = (
    'You are a visual fact-checker examining an image from the Arab world.\n'
    'Below are THREE statements about this image. '
    'Exactly ONE statement is grounded in the image (True). '
    'The other two are plausible-sounding hallucinations (False).\n\n'
    'Statement 1: {s0}\n'
    'Statement 2: {s1}\n'
    'Statement 3: {s2}\n\n'
    'Instructions:\n'
    '- Study the image carefully.\n'
    '- On the VERY FIRST line write ONLY: "Answer: X" where X is 1, 2, or 3.\n'
    '- Then explain step by step why that statement is grounded '
    'and why the other two are hallucinations.\n'
    'Do not write anything before the Answer line.'
)

# ── Fallback: individual CoT (only when joint parse fails, <5% expected) ──────
FALLBACK_PROMPT = (
    'You are a visual fact-checker.\n'
    'Decide if the following statement about the image is True or False.\n'
    'Context: exactly one of three statements about this image is True.\n\n'
    'Statement: "{s}"\n\n'
    'Write your answer on the FIRST line as exactly one word: True or False.\n'
    'Then briefly explain your reasoning.'
)


@torch.no_grad()
def vlm_call(image_path, text, max_new_tokens=256):
    """Single VLM forward pass, returns decoded string."""
    conv = [{'role': 'user', 'content': [
        {'type': 'image', 'image': image_path},
        {'type': 'text',  'text':  text}]}]
    prompt = processor.apply_chat_template(conv, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(conv)
    inputs = processor(text=[prompt], images=image_inputs, videos=video_inputs,
                       padding=True, return_tensors='pt').to(model.device)
    gen = model.generate(**inputs, max_new_tokens=max_new_tokens,
                     do_sample=False, num_beams=NUM_BEAMS)
    trimmed = gen[0][inputs.input_ids.shape[1]:]
    out = processor.decode(trimmed, skip_special_tokens=True).strip()
    del inputs, gen
    torch.cuda.empty_cache()
    return out


def parse_joint_answer(raw):
    """
    Parse 'Answer: X' from the model output (X in {1,2,3}).
    Run 4: checks FIRST non-empty line first (model instructed to answer first),
    then falls back to scanning the full text.
    Returns int (1-indexed) or None.
    """
    lines = [l.strip() for l in raw.splitlines() if l.strip()]
    # Check first line first (answer-first format)
    for line in lines:
        m = re.search(r'answer\s*[:\-]?\s*([123])', line, re.IGNORECASE)
        if m:
            return int(m.group(1))
        break  # only check first non-empty line with strict pattern
    # Broader scan over entire output
    for line in lines:
        m = re.search(r'answer\s*[:\-]?\s*([123])', line, re.IGNORECASE)
        if m:
            return int(m.group(1))
    # Last resort: any standalone digit 1/2/3 on its own line
    for line in lines:
        m = re.fullmatch(r'([123])', line)
        if m:
            return int(m.group(1))
    return None


def predict_item(image_path, statements):
    """
    Answer-first joint prediction for one item.
    Returns (labels list, raw response string, mode string).
    """
    # ── Joint answer-first pass ───────────────────────────────────────────────
    text = JOINT_PROMPT.format(s0=statements[0], s1=statements[1], s2=statements[2])
    raw  = vlm_call(image_path, text, max_new_tokens=MAX_NEW_TOKENS)
    chosen = parse_joint_answer(raw)

    if chosen is not None:
        labels = ['false', 'false', 'false']
        labels[chosen - 1] = 'true'   # 1-indexed → 0-indexed
        return labels, raw, 'joint'

    # ── Fallback: individual pass per statement ───────────────────────────────
    # Only reached if the model didn't output 'Answer: X' anywhere.
    labels = []
    fallback_raws = [raw]
    for stmt in statements:
        fb_raw = vlm_call(image_path, FALLBACK_PROMPT.format(s=stmt), max_new_tokens=128)
        fallback_raws.append(fb_raw)
        lines = [l.strip() for l in fb_raw.splitlines() if l.strip()]
        pred = None
        # Answer-first: check first line
        if lines:
            pred = evaluate_tf(lines[0]).pred
        if pred is None:
            pred = evaluate_tf(fb_raw).pred or 'false'
        labels.append(pred)

    # Enforce exactly-one-True constraint
    if labels.count('true') != 1:
        labels = ['false', 'false', 'false']
        labels[0] = 'true'

    return labels, ' ||| '.join(fallback_raws), 'fallback'


print('Answer-first inference function defined.')
print()
print('PROMPT PREVIEW (answer-first):')
print('─' * 60)
print(JOINT_PROMPT.format(s0='<statement 1>', s1='<statement 2>', s2='<statement 3>'))
print('─' * 60)


## 7. Run inference


In [ ]:
rows = []   # (id, statement_index, raw, prediction)
n_joint    = 0
n_fallback = 0

for r in tqdm(records, desc=f'[{RUN_ID}] joint infer'):
    labels, raw, mode = predict_item(paths[r['image']], r['statements'])
    if mode == 'joint':
        n_joint += 1
    else:
        n_fallback += 1
    for si, lbl in enumerate(labels):
        rows.append((r['id'], si, raw if si == 0 else '', lbl))

print(f'Done: {len(records)} items | joint: {n_joint} | fallback: {n_fallback}')
print(f'Fallback rate: {n_fallback / len(records) * 100:.1f}%  '
      f'(Run 2 baseline: <5%)')
print(f'Total rows: {len(rows)}')


## 8. Write predictions CSV


In [ ]:
import csv
OUT_CSV = f'predictions_{RUN_ID}_{LANG}.csv'
with open(OUT_CSV, 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f)
    w.writerow(['id', 'statement_index', 'raw_prediction', 'prediction_parsed'])
    w.writerows(rows)
print('Wrote', OUT_CSV, ':', len(rows), 'rows')


## 9. Score against dev labels

Change `SPLIT = 'dev'` in the config cell to run against labelled data.
Results here appear blank for `devtest` (blind split).

**Run 2 reference scores (dev split):**
| CI ↓ | Comb Acc ↑ | CFHR ↓ | Q+ Acc ↑ | Q- Acc ↑ |
|:---:|:---:|:---:|:---:|:---:|
| 0.092 | 0.908 | 0.000 | 0.908 | 0.954 |


In [ ]:
gold = {r['id']: r['labels'].index(True) for r in records if 'labels' in r}
if gold:
    by_item = {}
    for iid, si, _raw, parsed in rows:
        by_item.setdefault(iid, {})[si] = parsed

    total = q_plus = q_minus = q_minus_total = combined = 0
    n_partial = n_consistent = 0
    cfhr_num = cfhr_den = 0

    for iid, true_idx in gold.items():
        total += 1
        q_minus_total += 2
        pr = by_item.get(iid, {})
        labels = {i: evaluate_tf(pr.get(i, '')).pred for i in range(3)}
        ok_t = labels.get(true_idx) == 'true'
        ok_f = [labels.get(i) == 'false' for i in range(3) if i != true_idx]
        if ok_t: q_plus += 1
        q_minus += sum(ok_f)
        all_ok = ok_t and all(ok_f)
        any_ok = ok_t or any(ok_f)
        if all_ok: combined += 1
        if any_ok:
            n_partial += 1
            if all_ok: n_consistent += 1
        if ok_t:
            cfhr_den += 1
            if not all(ok_f): cfhr_num += 1

    ci           = 1 - n_consistent / n_partial if n_partial else 0.0
    cfhr         = cfhr_num / cfhr_den if cfhr_den else 0.0
    combined_acc = combined / total
    q_plus_acc   = q_plus / total
    q_minus_acc  = q_minus / q_minus_total

    print(f'Run: {RUN_ID}')
    print(f'Items: {total}')
    print()
    print(f'{'Metric':<35} {'Run 4':>10}   {'Run 2 ref':>10}')
    print('-' * 58)
    print(f'{'Contrastive Instability (CI) ↓':<35} {ci:>10.4f}   {'0.0920':>10}')
    print(f'{'Combined Accuracy ↑':<35} {combined_acc:>10.4f}   {'0.9080':>10}')
    print(f'{'CFHR ↓':<35} {cfhr:>10.4f}   {'0.0000':>10}')
    print(f'{'Q+ Accuracy ↑':<35} {q_plus_acc:>10.4f}   {'0.9080':>10}')
    print(f'{'Q- Accuracy ↑':<35} {q_minus_acc:>10.4f}   {'0.9540':>10}')
    print()
    delta_ci = ci - 0.0920
    direction = 'better ↓' if delta_ci < 0 else 'worse ↑'
    print(f'CI delta vs Run 2: {delta_ci:+.4f}  ({direction})')
else:
    print(f"'{SPLIT}' is blind — set SPLIT='dev' to score locally.")
    print(f'Submit prediction_{RUN_ID}_{LANG}.zip to Codabench for devtest score.')


## 10. Build Codabench submission zip


In [ ]:
import zipfile, csv
with open('prediction.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f)
    w.writerow(['id', 'statement_index', 'prediction'])
    for iid, si, _raw, parsed in rows:
        w.writerow([iid, si, parsed])
zip_name = f'prediction_{RUN_ID}_{LANG}.zip'
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write('prediction.csv', 'prediction.csv')
print(f'Wrote {zip_name}  →  upload this to Codabench')
print(f'Leaderboard: https://www.codabench.org/competitions/17051  (1b English)')
